In [ ]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import json

food_paquet = "C:/Users/Administrateur/Documents/Nutriscore/data/food.parquet"

In [21]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# pq.read_pandas('data/food.parquet', columns=['lang']).to_pandas()
# pq.read_pandas('data/food.parquet', columns=['lang'], filters=[('lang', '==', 'fr')]).to_pandas()

# pq.read_pandas('data/food.parquet', columns=['nutriments']).to_pandas().head(5)

#pq.read_schema('data/food.parquet').names
#pq.read_schema('data/food.parquet')
# pq.read_schema('data/food.parquet').field('nutriments').type

# open_food_facts_df = pd.read_parquet(food_paquet, columns=("countries_tags"))
# open_food_facts_df = pq.read_table(food_paquet, columns=["countries_tags", "nutriments"])

# colonnes = open_food_facts_df.column_names
# type_colonnes = open_food_facts_df.field("nutriments").type

# print(colonnes)
# print(type_colonnes)

# print(open_food_facts_df)

# non utilisé on va faire plutot des fonctions par type de colonnes qu'on a au préalable choisis
# def save_set_unique(col: str):

#     ar_col = pq.read_table(food_paquet, columns=[col])
#     t = ar_col.schema.field(col).type
#     val_list = ar_col.to_pylist()

#     result = set()

#     if pa.types.is_list(t) or pa.types.is_large_list(t):
#         for line in val_list:
#             if line is not None:
#                 for element in line:
#                     if element is not None:
#                         if isinstance(element, dict): 
#                             for valeur in element:
#                                 result.add(valeur)
#                         else:
#                             result.add(element)                    
#         return result


# Fonction récursive qui affiche le type et structure d'un champ pyarrow

def describe_field(field: pa.Field, indent: int = 0):

    prefix = "  " * indent
    t = field.type

    if pa.types.is_list(t) or pa.types.is_large_list(t):
        print(f"{prefix}{field.name}: LIST de")
        describe_field(t.value_field, indent + 1)

    elif pa.types.is_struct(t):
        print(f"{prefix}{field.name}: STRUCT avec les champs:")
        for sub_field in t:
            describe_field(sub_field, indent + 1)

    else:
        print(f"{prefix}{field.name}: {t}")

# Parcourt toutes les colonnes d'un fichier parquet et affiche leur type et structure grace à describe field
def describe_schema(path: str):
    schema = pq.read_schema(path)
    print(f"Fichier: {path} — {len(schema)} colonnes\n")
    for field in schema:
        describe_field(field)
        print()

describe_schema(food_paquet)

Fichier: C:/Users/Administrateur/Documents/Nutriscore/data/food.parquet — 111 colonnes

additives_n: int32

additives_tags: LIST de
  element: string

allergens_tags: LIST de
  element: string

brands_tags: LIST de
  element: string

brands: string

categories: string

categories_tags: LIST de
  element: string

categories_properties: STRUCT avec les champs:
  ciqual_food_code: int32
  agribalyse_food_code: int32
  agribalyse_proxy_food_code: int32

checkers_tags: LIST de
  element: string

ciqual_food_name_tags: LIST de
  element: string

cities_tags: LIST de
  element: string

code: string

compared_to_category: string

complete: int32

completeness: float

correctors_tags: LIST de
  element: string

countries_tags: LIST de
  element: string

created_t: int64

creator: string

data_quality_errors_tags: LIST de
  element: string

data_quality_info_tags: LIST de
  element: string

data_quality_warnings_tags: LIST de
  element: string

data_sources_tags: LIST de
  element: string

envir

In [ ]:

def verif_col_string(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    for valeur in val_list:
        # print(type(valeur[col]))
        if valeur[col] is None:
            null_compt += 1
        # if not isinstance(valeur[col], str):
        #     print(valeur[col])
        #     type_compt += 1
    # print(len(val_list))
    return null_compt

verif_col_string("brands")

1595058

In [70]:
def verif_col_codeb(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    false_compt = 0
    for valeur in val_list:
        if valeur[col] is None:
            null_compt += 1
        if len(valeur[col]) > 13 or len(valeur[col]) < 13:
            false_compt += 1
        # if not isinstance(valeur[col], str):
        #     print(valeur[col])
    # print(len(val_list))
    return null_compt, false_compt

null_compt, false_compt = verif_col_codeb("code")
print(f"valeur nulle : {null_compt} valeur fausse : {false_compt}")

valeur nulle : 0 valeur fausse : 323624


In [71]:
def verif_col_complet(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    false_compt = 0
    for valeur in val_list:
        # print(type(valeur[col]))
        if valeur[col] is None:
            null_compt += 1
        if valeur[col] is not None and (valeur[col] > 1 or valeur[col] == 0):
            false_compt += 1
    # print(len(val_list))
    return null_compt, false_compt

null_compt, typerr_compt = verif_col_complet("completeness")
print(f"valeur nulle : {null_compt} erreur de type : {typerr_compt}")

valeur nulle : 18 erreur de type : 16806


In [ ]:
def verif_col_countrytag(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    typerr_compt = 0
    for valeur in val_list:
        if valeur[col] is None:
            null_compt += 1
        else:
            for item in valeur[col]:
                # print(type(item))
                if not isinstance(item, str):
                    typerr_compt += 1
    # print(len(val_list))
    return null_compt, typerr_compt

null_compt, typerr_compt = verif_col_countrytag("countries_tags")
print(f"valeur nulle : {null_compt} erreur de type : {typerr_compt}")

(9741, 0)

In [69]:
def verif_col_data_quality(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    typerr_compt = 0
    for valeur in val_list:
        if valeur[col] is None:
            null_compt += 1
        else:
            for item in valeur[col]:
                # print(type(item))
                if not isinstance(item, str):
                    typerr_compt += 1
    # print(len(val_list))
    return null_compt, typerr_compt

null_compt, typerr_compt = verif_col_data_quality("data_quality_errors_tags")
print(f"valeur nulle : {null_compt} erreur de type : {typerr_compt}")

valeur nulle : 43069 erreur de type : 0


In [ ]:
import re

def is_date_valide(date_str):
    # passage au regex division par 4 du temps d'éxécution
    return bool(re.match(r"^\d{4}(-\d{2}(-\d{2})?)?$", date_str))
    # formats = ["%Y-%m-%d", "%Y-%m", "%Y"]
    # for fmt in formats:
    #     try:
    #         datetime.strptime(date_str, fmt)
    #         return True
    #     except ValueError:
    #         continue
    # return False

def verif_col_entry_dates(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    typerr_compt = 0
    daterr_compt = 0
    for valeur in val_list:
        if valeur[col] is None:
            null_compt += 1
        else:
            for item in valeur[col]:
                if not isinstance(item, str):
                    typerr_compt += 1
                if not is_date_valide(item):
                    daterr_compt += 1
    # print(len(val_list))
    return null_compt, typerr_compt, daterr_compt

null_compt, typerr_compt, daterr_compt = verif_col_entry_dates("entry_dates_tags")
print(f"valeur nulle : {null_compt} erreur de type : {typerr_compt} erreur de date : {daterr_compt}")

valeur nulle : 4 erreur de type : 0 erreur de date : 2717


In [77]:
def verif_col_food_groups(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    typerr_compt = 0
    for valeur in val_list:
        if valeur[col] is None:
            null_compt += 1
        else:
            for item in valeur[col]:
                # print(type(item))
                if not isinstance(item, str):
                    typerr_compt += 1
    # print(len(val_list))
    return null_compt, typerr_compt

null_compt, typerr_compt = verif_col_data_quality("food_groups_tags")
print(f"valeur nulle : {null_compt} erreur de type : {typerr_compt}")

valeur nulle : 43426 erreur de type : 0
